# FASS Component Score Benchmark

Computes per-image SSIM, Spearman, Jaccard, and composite FASS scores for post-hoc attribution methods under prediction-invariant perturbations.

**To use:** edit Section 1 only. Run all cells in order.

In [ ]:
import subprocess, sys
!pip install -q 'numpy>=1.24,<2.0' captum scikit-image --upgrade
print(subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                     capture_output=True, text=True).stdout.strip())
print('If numpy version changed: Runtime > Restart session, then skip this cell.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 164.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 168.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; 

## Section 1: Configuration
Edit this section only.

In [ ]:
import os, gc, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from io import BytesIO
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from captum.attr import IntegratedGradients, GradientShap, LayerGradCam

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/xai-stability-benchmark'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
NORMALIZE     = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)

PERTURBATIONS   = ['rotation', 'noise', 'translation', 'brightness', 'jpeg']
MAX_IMAGES      = 10000
ATTR_BATCH_SIZE = 8
FILTER_BATCH    = 64
TOP_K           = 100
NUM_WORKERS     = 4


def build_models():
    reg = {}
    m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1).eval()
    reg['resnet50'] = {'model': m, 'gradcam_layer': m.layer4[-1]}
    m = tvm.densenet121(weights=tvm.DenseNet121_Weights.IMAGENET1K_V1).eval()
    reg['densenet121'] = {'model': m, 'gradcam_layer': m.features[-1]}
    m = tvm.convnext_tiny(weights=tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1).eval()
    reg['convnext_tiny'] = {'model': m, 'gradcam_layer': m.features[-1]}
    m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.IMAGENET1K_V1).eval()
    reg['vit_b_16'] = {'model': m, 'gradcam_layer': m.encoder.layers[-1]}
    return reg


MODEL_REGISTRY = build_models()

DATASET_REGISTRY = {
    'imagenet': {
        'path': os.path.join(DRIVE_BASE, 'datasets/imagenet_sample.parquet'),
        'type': 'parquet',
        'result_dir': os.path.join(DRIVE_BASE, 'results_imagenet_components'),
    },
    'cifar10': {
        'path': os.path.join(DRIVE_BASE, 'datasets/cifar_sample.npz'),
        'type': 'npz',
        'result_dir': os.path.join(DRIVE_BASE, 'results_cifar10_components'),
    },
    'coco': {
        'path': '/content/drive/MyDrive/xai-stability-data/coco/train2017',
        'type': 'directory',
        'result_dir': os.path.join(DRIVE_BASE, 'results_coco_components'),
    },
}

for cfg in DATASET_REGISTRY.values():
    os.makedirs(cfg['result_dir'], exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Models:   {list(MODEL_REGISTRY.keys())}')
print(f'Datasets: {list(DATASET_REGISTRY.keys())}')

Mounted at /content/drive
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 319MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 392MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 354MB/s]


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:00<00:00, 366MB/s]


Device: cuda
Models:   ['resnet50', 'densenet121', 'convnext_tiny', 'vit_b_16']
Datasets: ['imagenet', 'cifar10', 'coco']


## Section 2: Dataset and Perturbations

In [ ]:
def _jpeg_perturb(img):
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=40)
    buf.seek(0)
    return NORMALIZE(TF.to_tensor(Image.open(buf).convert('RGB')))


PERTURBATION_FNS = {
    'rotation':    lambda img: NORMALIZE(TF.to_tensor(TF.rotate(img, 15))),
    'translation': lambda img: NORMALIZE(TF.to_tensor(
                       TF.affine(img, angle=0, translate=(20, 0), scale=1.0, shear=0))),
    'brightness':  lambda img: NORMALIZE(TF.to_tensor(TF.adjust_brightness(img, 1.5))),
    'noise':       lambda img: NORMALIZE(torch.clamp(
                       TF.to_tensor(img) + torch.randn(3, 224, 224) * 0.15, 0, 1)),
    'jpeg':        _jpeg_perturb,
}


class PairDataset(Dataset):
    def __init__(self, images, perturbation):
        self.images = images
        self.p_fn   = PERTURBATION_FNS[perturbation]

    def __len__(self):
        return len(self.images)

    def _to_pil(self, x):
        if isinstance(x, np.ndarray):
            return Image.fromarray(x).convert('RGB')
        return x.convert('RGB') if isinstance(x, Image.Image) else x

    def __getitem__(self, idx):
        img  = TF.resize(self._to_pil(self.images[idx]), (224, 224))
        orig = NORMALIZE(TF.to_tensor(img))
        return torch.stack([orig, self.p_fn(img)]), idx


def load_images(cfg):
    dtype, path = cfg['type'], cfg['path']
    if dtype == 'npz':
        data = np.load(path)
        key = next((k for k in ['images', 'x', 'data'] if k in data), None)
        if key is None:
            key = next((k for k in data if data[k].ndim >= 3), None)
        if key is None:
            raise ValueError(f'No suitable array found in {path}. Keys: {list(data.keys())}')
        return list(data[key][:MAX_IMAGES])
    if dtype == 'parquet':
        df = pd.read_parquet(path).head(MAX_IMAGES)
        out = []
        for raw in df['image_bytes']:
            b = raw['bytes'] if isinstance(raw, dict) else raw
            out.append(Image.open(BytesIO(b)).convert('RGB'))
        return out
    if dtype == 'directory':
        files = sorted(f for f in os.listdir(path)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png')))[:MAX_IMAGES]
        return [os.path.join(path, f) for f in files]  # paths only, load lazily
    raise ValueError(f'Unknown dataset type: {dtype}')


class PairDataset(Dataset):
    def __init__(self, images, perturbation):
        self.images = images
        self.p_fn   = PERTURBATION_FNS[perturbation]

    def __len__(self):
        return len(self.images)

    def _load(self, x):
        if isinstance(x, str):                        # file path — load lazily
            return Image.open(x).convert('RGB')
        if isinstance(x, np.ndarray):
            return Image.fromarray(x).convert('RGB')
        return x.convert('RGB') if isinstance(x, Image.Image) else x

    def __getitem__(self, idx):
        img  = TF.resize(self._load(self.images[idx]), (224, 224))
        orig = NORMALIZE(TF.to_tensor(img))
        return torch.stack([orig, self.p_fn(img)]), idx


def make_loader(images, perturbation, batch_size=FILTER_BATCH):
    return DataLoader(PairDataset(images, perturbation),
                      batch_size=batch_size, num_workers=NUM_WORKERS,
                      pin_memory=True, drop_last=False)

## Section 3: FASS Metrics

In [ ]:
def _minmax(x):
    b = x.shape[0]
    f = x.view(b, -1)
    s = (b, 1, 1, 1) if x.dim() == 4 else (b, 1, 1)
    mn = f.min(1, keepdim=True)[0].view(s)
    mx = f.max(1, keepdim=True)[0].view(s)
    return (x - mn) / (mx - mn + 1e-8)


def ssim(a1, a2):
    a1, a2 = _minmax(a1), _minmax(a2)
    pool   = nn.AvgPool2d(11, stride=1, padding=5)
    m1, m2 = pool(a1), pool(a2)
    s12    = pool(a1 * a2) - m1 * m2
    s1     = pool(a1 * a1) - m1 * m1
    s2     = pool(a2 * a2) - m2 * m2
    c1, c2 = 0.01**2, 0.03**2
    return (((2*m1*m2+c1)*(2*s12+c2)) / ((m1**2+m2**2+c1)*(s1+s2+c2))
            ).view(a1.shape[0], -1).mean(1)


def spearman(a1, a2):
    b  = a1.shape[0]
    r1 = torch.argsort(torch.argsort(a1.view(b, -1), 1), 1).float()
    r2 = torch.argsort(torch.argsort(a2.view(b, -1), 1), 1).float()
    m1, m2 = r1.mean(1, keepdim=True), r2.mean(1, keepdim=True)
    num    = ((r1 - m1) * (r2 - m2)).sum(1)
    den    = torch.sqrt(((r1-m1)**2).sum(1) * ((r2-m2)**2).sum(1)) + 1e-8
    return (num / den + 1) / 2


def jaccard(a1, a2, k=TOP_K):
    b = a1.shape[0]
    _, i1 = torch.topk(a1.view(b, -1), k, 1)
    _, i2 = torch.topk(a2.view(b, -1), k, 1)
    return torch.stack([
        (inter := torch.isin(i1[i], i2[i]).sum().float()) / (2*k - inter)
        for i in range(b)
    ])


def fass_components(a1, a2):
    s, r, j = ssim(a1, a2), spearman(a1, a2), jaccard(a1, a2)
    return s, r, j, (s + r + j) / 3

## Section 4: Benchmark Engine

In [ ]:
def save_rows(rows, path):
    pd.DataFrame(rows).to_csv(path, mode='a', header=not os.path.exists(path), index=False)


def filter_retained(model, loader):
    buckets = []
    with torch.no_grad():
        for stack, idxs in loader:
            o, p = stack[:, 0].to(DEVICE), stack[:, 1].to(DEVICE)
            po   = torch.argmax(model(o), 1)
            mask = po == torch.argmax(model(p), 1)
            if mask.any():
                buckets.append((o[mask].cpu(), p[mask].cpu(),
                                po[mask].cpu(), idxs[mask.cpu()]))
            del o, p, po, mask
            torch.cuda.empty_cache()
    if not buckets:
        return None, None, None, None
    return tuple(torch.cat([b[i] for b in buckets]) for i in range(4))


def run_method(name, attr_fn, origs, perts, targets, store, pbar=None):
    for s in range(0, len(origs), ATTR_BATCH_SIZE):
        e = min(s + ATTR_BATCH_SIZE, len(origs))
        o = origs[s:e].to(DEVICE).requires_grad_(True)
        p = perts[s:e].to(DEVICE).requires_grad_(True)
        t = targets[s:e].to(DEVICE)
        try:
            a1, a2 = attr_fn(o, p, t)
            ss, rr, jj, ff = fass_components(a1, a2)
            for i, gi in enumerate(range(s, e)):
                store[gi].update({
                    f'{name}_ssim':     round(ss[i].item(), 6),
                    f'{name}_spearman': round(rr[i].item(), 6),
                    f'{name}_jaccard':  round(jj[i].item(), 6),
                    f'{name}_fass':     round(ff[i].item(), 6),
                })
            del a1, a2, ss, rr, jj, ff
        except RuntimeError:
            pass
        finally:
            del o, p, t
            torch.cuda.empty_cache()
        if pbar:
            pbar.update(1)
            pbar.set_postfix(method=name, refresh=False)


def build_attr_fns(model, gradcam_layer):
    ig  = IntegratedGradients(model)
    gs  = GradientShap(model)
    fns = {
        'ig':   lambda o, p, t: (ig.attribute(o, target=t),
                                 ig.attribute(p, target=t)),
        'shap': lambda o, p, t: (
            gs.attribute(o, baselines=torch.zeros_like(o), target=t, n_samples=5),
            gs.attribute(p, baselines=torch.zeros_like(p), target=t, n_samples=5)),
    }
    if gradcam_layer is not None:
        gcam = LayerGradCam(model, gradcam_layer)
        fns['gradcam'] = lambda o, p, t: (
            TF.resize(gcam.attribute(o, target=t), (224, 224)),
            TF.resize(gcam.attribute(p, target=t), (224, 224)))
    return fns


def run_benchmark(model_name, model_cfg, ds_name, images, perturbation, result_dir, pbar=None):
    path = os.path.join(result_dir, f'{model_name}_fass_components_{perturbation}.csv')
    done = path + '.done'

    if os.path.exists(done):
        tqdm.write(f'  [{ds_name}] {model_name}/{perturbation} — already done, skipping.')
        if pbar:
            pbar.update(1)
            pbar.set_postfix(skipped=f'{model_name}/{perturbation}', refresh=False)
        return

    start = 0
    if os.path.exists(path):
        try:
            ex = pd.read_csv(path)
            if not ex.empty:
                start = int(ex['image_index'].max()) + 1
        except Exception:
            pass

    model = model_cfg['model']
    model.to(DEVICE)
    total = len(images)

    if start >= total:
        open(done, 'w').close()
        model.cpu()
        if pbar:
            pbar.update(1)
        return

    tqdm.write(f'  [{ds_name}] {model_name}/{perturbation} (start={start})')
    loader = make_loader(images[start:], perturbation)
    origs, perts, targets, idxs = filter_retained(model, loader)

    if origs is None:
        tqdm.write(f'0 retained pairs — skipping.')
        open(done, 'w').close()
        model.cpu()
        if pbar:
            pbar.update(1)
        return

    tqdm.write(f'{len(origs)}/{total - start} retained.')
    store    = {i: {'image_index': idxs[i].item() + start} for i in range(len(origs))}
    attr_fns = build_attr_fns(model, model_cfg['gradcam_layer'])

    for name, fn in attr_fns.items():
        if pbar:
            pbar.set_postfix(model=model_name, p=perturbation, method=name, refresh=False)
        run_method(name, fn, origs, perts, targets, store, pbar=None)
        torch.cuda.empty_cache()
        gc.collect()

    expected = [f'{m}_fass' for m in attr_fns]
    rows     = [v for v in store.values() if all(k in v for k in expected)]
    if rows:
        save_rows(rows, path)
        open(done, 'w').close()
        tqdm.write(f'Saved {len(rows)} rows.')
    else:
        tqdm.write(f'No rows saved — check ATTR_BATCH_SIZE.')

    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()

    if pbar:
        pbar.update(1)
        pbar.set_postfix(done=f'{model_name}/{perturbation}', refresh=False)

## Section 5: Run

In [ ]:
import math

total_conditions = len(DATASET_REGISTRY) * len(MODEL_REGISTRY) * len(PERTURBATIONS)

_image_cache = {}

with tqdm(total=total_conditions, desc='FASS', unit='condition') as pbar:
    for ds_name, ds_cfg in DATASET_REGISTRY.items():
        tqdm.write(f'\n{ds_name.upper()}')
        if ds_name not in _image_cache:
            _image_cache[ds_name] = load_images(ds_cfg)
        images = _image_cache[ds_name]
        for model_name, model_cfg in MODEL_REGISTRY.items():
            for p_type in PERTURBATIONS:
                run_benchmark(
                    model_name, model_cfg, ds_name, images,
                    p_type, ds_cfg['result_dir'], pbar=pbar
                )
        del _image_cache[ds_name]
        gc.collect()

tqdm.write('\nAll FASS conditions complete.')

FASS:   0%|          | 0/60 [00:00<?, ?condition/s]


IMAGENET


FASS:  33%|███▎      | 20/60 [02:03<1:22:14, 123.37s/condition, skipped=vit_b_16/jpeg]

  [imagenet] resnet50/rotation — already done, skipping.
  [imagenet] resnet50/noise — already done, skipping.
  [imagenet] resnet50/translation — already done, skipping.
  [imagenet] resnet50/brightness — already done, skipping.
  [imagenet] resnet50/jpeg — already done, skipping.
  [imagenet] densenet121/rotation — already done, skipping.
  [imagenet] densenet121/noise — already done, skipping.
  [imagenet] densenet121/translation — already done, skipping.
  [imagenet] densenet121/brightness — already done, skipping.
  [imagenet] densenet121/jpeg — already done, skipping.
  [imagenet] convnext_tiny/rotation — already done, skipping.
  [imagenet] convnext_tiny/noise — already done, skipping.
  [imagenet] convnext_tiny/translation — already done, skipping.
  [imagenet] convnext_tiny/brightness — already done, skipping.
  [imagenet] convnext_tiny/jpeg — already done, skipping.
  [imagenet] vit_b_16/rotation — already done, skipping.
  [imagenet] vit_b_16/noise — already done, skipping.


FASS:  67%|██████▋   | 40/60 [02:06<01:26,  4.32s/condition, skipped=vit_b_16/jpeg]

  [cifar10] resnet50/rotation — already done, skipping.
  [cifar10] resnet50/noise — already done, skipping.
  [cifar10] resnet50/translation — already done, skipping.
  [cifar10] resnet50/brightness — already done, skipping.
  [cifar10] resnet50/jpeg — already done, skipping.
  [cifar10] densenet121/rotation — already done, skipping.
  [cifar10] densenet121/noise — already done, skipping.
  [cifar10] densenet121/translation — already done, skipping.
  [cifar10] densenet121/brightness — already done, skipping.
  [cifar10] densenet121/jpeg — already done, skipping.
  [cifar10] convnext_tiny/rotation — already done, skipping.
  [cifar10] convnext_tiny/noise — already done, skipping.
  [cifar10] convnext_tiny/translation — already done, skipping.
  [cifar10] convnext_tiny/brightness — already done, skipping.
  [cifar10] convnext_tiny/jpeg — already done, skipping.
  [cifar10] vit_b_16/rotation — already done, skipping.
  [cifar10] vit_b_16/noise — already done, skipping.
  [cifar10] vit_b

FASS: 100%|██████████| 60/60 [03:40<00:00,  3.68s/condition, skipped=vit_b_16/jpeg]      

  [coco] resnet50/rotation — already done, skipping.
  [coco] resnet50/noise — already done, skipping.
  [coco] resnet50/translation — already done, skipping.
  [coco] resnet50/brightness — already done, skipping.
  [coco] resnet50/jpeg — already done, skipping.
  [coco] densenet121/rotation — already done, skipping.
  [coco] densenet121/noise — already done, skipping.
  [coco] densenet121/translation — already done, skipping.
  [coco] densenet121/brightness — already done, skipping.
  [coco] densenet121/jpeg — already done, skipping.
  [coco] convnext_tiny/rotation — already done, skipping.
  [coco] convnext_tiny/noise — already done, skipping.
  [coco] convnext_tiny/translation — already done, skipping.
  [coco] convnext_tiny/brightness — already done, skipping.
  [coco] convnext_tiny/jpeg — already done, skipping.
  [coco] vit_b_16/rotation — already done, skipping.
  [coco] vit_b_16/noise — already done, skipping.
  [coco] vit_b_16/translation — already done, skipping.
  [coco] vit

## Section 6: LIME

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='captum')

from captum.attr import Lime
from captum._utils.models.linear_model import SkLearnLinearRegression
from skimage.segmentation import quickshift

LIME_ATTR_BATCH = 4
LIME_N_SAMPLES  = 25
LIME_SEG_KWARGS = dict(kernel_size=4, max_dist=200, ratio=0.2)


def _make_quickshift_mask(batch_tensor):
    np.random.seed(SEED)
    imgs = batch_tensor.detach().cpu().permute(0, 2, 3, 1).numpy()
    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)
    imgs = (imgs * std + mean).clip(0, 1)
    masks = []
    for img in imgs:
        seg = quickshift(img, **LIME_SEG_KWARGS)
        masks.append(torch.tensor(seg, dtype=torch.long).unsqueeze(0))
    return torch.stack(masks).to(batch_tensor.device)


def _run_lime(model_name, model_cfg, ds_name, images, perturbation, result_dir):
    path = os.path.join(result_dir,
                        f'{model_name}_fass_components_lime_{perturbation}.csv')
    done = path + '.done'

    if os.path.exists(done):
        print(f'  [{ds_name}] {model_name}/{perturbation} LIME — already done, skipping.')
        return

    start = 0
    if os.path.exists(path):
        try:
            ex = pd.read_csv(path)
            if not ex.empty:
                start = int(ex['image_index'].max()) + 1
        except Exception:
            pass

    print(f'  [{ds_name}] {model_name}/{perturbation} LIME (resume from img {start})')

    model = model_cfg['model']
    model.to(DEVICE)
    total = len(images)

    if start >= total:
        open(done, 'w').close()
        model.cpu()
        return

    loader = make_loader(images[start:], perturbation)
    origs, perts, targets, idxs = filter_retained(model, loader)

    if origs is None:
        print('    0 retained pairs — skipping.')
        open(done, 'w').close()
        model.cpu()
        return

    n = len(origs)
    print(f'    {n}/{total - start} pairs retained ({100*n/(total-start):.1f}%)')

    lime      = Lime(model, interpretable_model=SkLearnLinearRegression())
    row_store = {i: {'image_index': idxs[i].item() + start} for i in range(n)}

    for s in tqdm(range(0, n, LIME_ATTR_BATCH),
                  desc=f'    {model_name}/{perturbation}', unit='batch'):
        e = min(s + LIME_ATTR_BATCH, n)
        o = origs[s:e].to(DEVICE).requires_grad_(True)
        p = perts[s:e].to(DEVICE).requires_grad_(True)
        t = targets[s:e].to(DEVICE)
        try:
            mask_o = _make_quickshift_mask(o)
            mask_p = _make_quickshift_mask(p)
            a1 = lime.attribute(o, target=t,
                                feature_mask=mask_o,
                                n_samples=LIME_N_SAMPLES,
                                perturbations_per_eval=LIME_N_SAMPLES)
            a2 = lime.attribute(p, target=t,
                                feature_mask=mask_p,
                                n_samples=LIME_N_SAMPLES,
                                perturbations_per_eval=LIME_N_SAMPLES)
            ss, rr, jj, ff = fass_components(a1, a2)
            for i, gi in enumerate(range(s, e)):
                row_store[gi].update({
                    'lime_ssim':     round(ss[i].item(), 6),
                    'lime_spearman': round(rr[i].item(), 6),
                    'lime_jaccard':  round(jj[i].item(), 6),
                    'lime_fass':     round(ff[i].item(), 6),
                })
            del a1, a2, ss, rr, jj, ff, mask_o, mask_p
        except RuntimeError:
            torch.cuda.empty_cache()
        finally:
            del o, p, t
            torch.cuda.empty_cache()

    rows = [v for v in row_store.values() if 'lime_fass' in v]
    if rows:
        save_rows(rows, path)
        open(done, 'w').close()
        print(f'    Saved {len(rows)} rows.')
    else:
        print('    Warning: no rows saved. Try reducing LIME_ATTR_BATCH.')

    del lime
    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()


# ── Run ────────────────────────────────────────────────────────────────────
for ds_name, ds_cfg in DATASET_REGISTRY.items():
    print(f'\n═══ {ds_name.upper()} — LIME ═══')
    images = load_images(ds_cfg)
    for model_name, model_cfg in MODEL_REGISTRY.items():
        for p_type in PERTURBATIONS:
            _run_lime(model_name, model_cfg, ds_name,
                      images, p_type, ds_cfg['result_dir'])
    del images
    gc.collect()

print('\nLIME complete.')


═══ IMAGENET — LIME ═══
  [imagenet] resnet50/rotation LIME — already done, skipping.
  [imagenet] resnet50/noise LIME — already done, skipping.
  [imagenet] resnet50/translation LIME — already done, skipping.
  [imagenet] resnet50/brightness LIME — already done, skipping.
  [imagenet] resnet50/jpeg LIME — already done, skipping.
  [imagenet] densenet121/rotation LIME — already done, skipping.
  [imagenet] densenet121/noise LIME — already done, skipping.
  [imagenet] densenet121/translation LIME — already done, skipping.
  [imagenet] densenet121/brightness LIME — already done, skipping.
  [imagenet] densenet121/jpeg LIME — already done, skipping.
  [imagenet] convnext_tiny/rotation LIME (resume from img 0)
    7977/10000 pairs retained (79.8%)


    convnext_tiny/rotation:  65%|██████▌   | 1299/1995 [58:02<29:27,  2.54s/batch]

## Section 7: Sanity Check

In [ ]:
ok = True
for ds_cfg in DATASET_REGISTRY.values():
    for model_name in MODEL_REGISTRY:
        for p_type in PERTURBATIONS:
            for prefix in ['', '_lime']:
                path = os.path.join(ds_cfg['result_dir'],
                                    f'{model_name}_fass_components{prefix}_{p_type}.csv')
                if not os.path.exists(path):
                    continue
                df = pd.read_csv(path)
                for m in {c.replace('_fass', '') for c in df.columns if c.endswith('_fass')}:
                    err = ((df[f'{m}_ssim'] + df[f'{m}_spearman'] + df[f'{m}_jaccard']) / 3
                           - df[f'{m}_fass']).abs().max()
                    if err >= 1e-4:
                        ok = False
                        print(f'MISMATCH {model_name} {p_type} {m} err={err:.2e}')
print('All checks passed.' if ok else 'Mismatches found.')

## Section 8: Aggregate

In [ ]:
for ds_name, ds_cfg in DATASET_REGISTRY.items():
    records = []
    for model_name in MODEL_REGISTRY:
        for p_type in PERTURBATIONS:
            for prefix in ['', '_lime']:
                path = os.path.join(ds_cfg['result_dir'],
                                    f'{model_name}_fass_components{prefix}_{p_type}.csv')
                if not os.path.exists(path):
                    continue
                df = pd.read_csv(path)
                for m in {c.replace('_fass', '') for c in df.columns if c.endswith('_fass')}:
                    records.append({
                        'model': model_name, 'perturbation': p_type, 'method': m,
                        'ssim':     df[f'{m}_ssim'].mean(),
                        'spearman': df[f'{m}_spearman'].mean(),
                        'jaccard':  df[f'{m}_jaccard'].mean(),
                        'fass':     df[f'{m}_fass'].mean(),
                        'n':        len(df),
                    })
    if not records:
        print(f'{ds_name}: no data.')
        continue
    agg = pd.DataFrame(records)
    agg.to_csv(os.path.join(ds_cfg['result_dir'], f'summary_{ds_name}.csv'), index=False)
    print(f'\n{ds_name.upper()} method averages')
    print(agg.groupby('method')[['ssim', 'spearman', 'jaccard', 'fass']].mean().round(3).to_string())
    print(f'\n{ds_name.upper()} per model')
    print(agg.groupby(['model', 'method'])[['ssim', 'spearman', 'jaccard', 'fass']].mean().round(3).to_string())

## Section 9: Status

In [ ]:
for ds_name, ds_cfg in DATASET_REGISTRY.items():
    print(f'\n{ds_name}')
    for model_name in MODEL_REGISTRY:
        for p_type in PERTURBATIONS:
            for prefix in ['', '_lime']:
                path = os.path.join(ds_cfg['result_dir'],
                                    f'{model_name}_fass_components{prefix}_{p_type}.csv')
                if not os.path.exists(path):
                    print(f'  {model_name:17s} {p_type:12s}{prefix:5s} MISSING')
                    continue
                df = pd.read_csv(path)
                ms = {c.replace('_fass', '') for c in df.columns if c.endswith('_fass')}
                s  = ' | '.join(f"{m}: {df[f'{m}_fass'].mean():.3f}" for m in sorted(ms))
                print(f'  {model_name:17s} {p_type:12s}{prefix:5s} {len(df):5d} | {s}')